# Lab 3, Student Performance Dataset 

Team Members: Sandro Juric & Scotty Seethoff

### AI usage disclaimer
We have used the Copilot in this project, primarily for code refactoring in VS Code

# Business Understanding

### Overview & purpose:

This is synthetically generated dataset of 5000 student with several key demographic data including age, gender, academic level.  Accompanied by several features we can use to predict classifies of focus index, burnout level, productivity score, and exam score. Assuming we can build a accurate prediction model that can determine these classes based on features of number of study hours, self-study hours, online classes, number of social media hours, gaming hours, sleep hours, screen time hours, excise amounts, caffeine intake, internet quality, and mental health score once we are able to obtain real time data we could better devise plans to help out students be more successful, productive, and increase their focus index, while keeping the burnout level low. Even though this data set was generated synthetically, data has realistic relationship between features that can be used to simulate real world scenarios.

### Prediction task:
Our prediction tasks are to take multi-class student classifications as input and predict features based on that data. We are seeking to predict exam scores, burn out level, and overall productivity score.  In order for this prediction model to be useful we'll need to have at least 85% accuracy at the exam score predictions and 80% accuracy at the burn out level and productivity score. Since we are dealing with synthetic dataset in order for this model to be truly tested we'll need to have it deployed and collect live data. Not only to obtain real life data but to monitor the dynamic nature of these classifiers as all three of the prediction classes are subject to sudden changes.

### Data Imporatance:
School administrators struggle on a daily basis to improve the overall education of all the students. It would be a great benefit to know what factors have the most effect on the overall student scores, their productivity, and their burn out level.  The educators have to balance the given the students enough study material to be successful versus overwhelming them with too material. If we can build a high confidence model this balancing task would be greatly simplified.

### Benchmark Test
As mentioned above we'll need to have at least 85% accuracy to achieve confidence in predicting exam scores. We'll use multi different algorithms to achieve that goal and will weigh accuracy with the amount of computing power that is needed.

### Original Dataset:
Dataset: https://www.kaggle.com/datasets/amar5693/student-performance-dataset

# Data Preparation

Importing the data from static offline CSV file. Removing unnecessary variables and converting gender, academic level, and internet quality to integer map. Examining the data structure and sample records. Finally, looking over the statistical analysis of the dataset to see if there are any outliers that will need additional data manipulation before proceeding with the prediction task.

In [5]:
import pandas as pd
import numpy as np

# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None  # default='warn'
# Ensure future behavior for downcasting is explicit
pd.set_option('future.no_silent_downcasting', True)

df = pd.read_csv('ultimate_student_productivity_dataset_5000.csv')

# select relevant columns
dfSelect = df[['student_id', 'age', 'gender', 'academic_level', 'study_hours', 'self_study_hours', 'online_classes_hours', 'social_media_hours', 'gaming_hours', 'sleep_hours', 'screen_time_hours', 'exercise_minutes', 'caffeine_intake_mg', 'internet_quality', 'mental_health_score', 'focus_index', 'burnout_level', 'productivity_score', 'exam_score']]

gender_map = {
    'Other': 0,
    'Male': 1,
    'Female': 2
}

academic_level_map = {
    'High School': 1,
    'Undergraduate': 2,
    'Graduate': 3,
    'PhD': 4
}

internet_quality_map = {
    'Poor': 1,
    'Average': 2,
    'Good': 3,
    'Excellent': 4 
}

dfSelect['gender'] = dfSelect['gender'].map(gender_map).fillna(0)
dfSelect['academic_level'] = dfSelect['academic_level'].map(academic_level_map).fillna(0)
dfSelect['internet_quality'] = dfSelect['internet_quality'].map(internet_quality_map).fillna(0)


dfSelect.info()
# format and display of the first few rows used the pandas Styler for better visualization in Jupyter Notebooks https://pandas.pydata.org/docs/user_guide/style.html
display(dfSelect.head(10).style.background_gradient(axis=None, cmap="YlGnBu"))

dfSelect.describe().style.background_gradient(cmap="YlOrRd")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   student_id            5000 non-null   int64  
 1   age                   5000 non-null   int64  
 2   gender                5000 non-null   int64  
 3   academic_level        5000 non-null   float64
 4   study_hours           5000 non-null   float64
 5   self_study_hours      5000 non-null   float64
 6   online_classes_hours  5000 non-null   float64
 7   social_media_hours    5000 non-null   float64
 8   gaming_hours          5000 non-null   float64
 9   sleep_hours           5000 non-null   float64
 10  screen_time_hours     5000 non-null   float64
 11  exercise_minutes      5000 non-null   int64  
 12  caffeine_intake_mg    5000 non-null   int64  
 13  internet_quality      5000 non-null   int64  
 14  mental_health_score   5000 non-null   int64  
 15  focus_index          

,student_id,age,gender,academic_level,study_hours,self_study_hours,online_classes_hours,social_media_hours,gaming_hours,sleep_hours,screen_time_hours,exercise_minutes,caffeine_intake_mg,internet_quality,mental_health_score,focus_index,burnout_level,productivity_score,exam_score
0,1,18,0,1.000000,7.640000,1.560000,2.200000,3.050000,2.190000,6.520000,6.470000,81,38,3,10,43.050000,31.770000,73.650000,50.160000
1,2,18,0,1.000000,2.210000,2.220000,2.100000,1.650000,2.550000,5.970000,6.050000,111,339,3,3,15.920000,37.000000,13.700000,1.000000
2,3,22,1,1.000000,3.450000,0.000000,0.290000,1.340000,2.080000,8.390000,7.620000,68,266,3,8,27.390000,34.370000,45.150000,18.300000
3,4,17,0,1.000000,5.750000,2.080000,3.010000,2.270000,2.200000,6.310000,11.670000,113,480,1,3,22.310000,77.310000,20.920000,9.370000
4,5,19,0,1.000000,6.830000,1.720000,3.330000,2.650000,0.700000,8.010000,10.020000,121,24,3,8,38.110000,39.530000,59.230000,27.810000
5,6,25,1,2.000000,2.210000,3.500000,1.690000,4.470000,1.560000,8.340000,8.060000,110,288,2,8,30.520000,33.640000,36.200000,18.530000
6,7,22,0,1.000000,4.600000,2.370000,1.270000,3.120000,1.890000,7.610000,11.340000,19,123,1,6,33.260000,62.010000,37.590000,10.350000
7,8,17,1,1.000000,8.770000,3.780000,2.090000,1.760000,3.130000,7.620000,7.270000,135,379,3,1,31.980000,45.980000,45.040000,19.890000
8,9,16,2,2.000000,6.600000,0.840000,1.000000,4.390000,0.850000,8.290000,8.240000,101,308,2,10,30.230000,49.440000,61.830000,30.730000
9,10,17,2,1.000000,3.980000,0.160000,1.290000,3.200000,2.130000,5.430000,7.790000,142,415,2,9,20.260000,60.780000,29.060000,8.990000


,student_id,age,gender,academic_level,study_hours,self_study_hours,online_classes_hours,social_media_hours,gaming_hours,sleep_hours,screen_time_hours,exercise_minutes,caffeine_intake_mg,internet_quality,mental_health_score,focus_index,burnout_level,productivity_score,exam_score
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,2500.500000,20.520400,0.995800,0.990800,4.539594,2.478734,2.011984,2.998086,1.564514,7.016492,6.979588,74.535600,251.450400,2.016400,5.507400,29.431616,45.615324,37.267716,18.803752
std,1443.520003,2.870406,0.810132,0.815873,1.821665,1.177990,0.983906,1.467949,1.110807,1.163692,2.486214,42.932293,143.842712,0.819918,2.869145,9.962902,14.246591,16.849397,12.130840
min,1.000000,16.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1250.750000,18.000000,0.000000,0.000000,3.250000,1.660000,1.320000,1.990000,0.670000,6.237500,5.280000,37.000000,129.000000,1.000000,3.000000,22.567500,35.727500,25.290000,9.337500
50%,2500.500000,20.000000,1.000000,1.000000,4.530000,2.480000,2.010000,2.980000,1.490000,7.010000,6.950000,75.000000,252.000000,2.000000,5.000000,29.430000,45.690000,36.860000,18.010000
75%,3750.250000,23.000000,2.000000,2.000000,5.760000,3.290000,2.690000,4.030000,2.340000,7.810000,8.710000,112.000000,376.000000,3.000000,8.000000,36.242500,55.352500,49.142500,27.400000
max,5000.000000,25.000000,2.000000,2.000000,11.840000,7.410000,6.000000,8.280000,5.640000,10.000000,15.300000,149.000000,499.000000,3.000000,10.000000,63.480000,97.580000,98.020000,64.090000


### Analysis

There are no gaps in the data all variables have data in on records and statistical data shows significant anomalies that would need to be addressed. The are some records with 0 values but over mean values indicate that those are not skewing the data in any significant way.

### Splitting the data into train and test

With the total number of rows of 5000 using the 80/20 split would yield 4000 rows for the training set and 1000 rows for the test sample set. 4000 rows should be sufficient for learning algorithms to learn meaningful patterns. With the dataset of 1000 for the test set this should provide us with a statistically sound evaluation set with a small sample error somewhere between 1 and 3 percent. Therefore, we can conclude that is appropriate to use the 80/20 split for this dataset.


In [8]:
#dividing data into training and testing sets
from sklearn.model_selection import train_test_split

# Define features (X) and target variable (y)
X = dfSelect.drop(columns=['productivity_score', 'exam_score', 'burnout_level'])
y = dfSelect[['productivity_score', 'exam_score', 'burnout_level']]

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Modeling

Took the existing models and added the solver attribute to the Logistic Regression class that will be used as a parameter to pick which regression model to use

In [ ]:
from scipy.special import expit
from numpy.linalg import pinv



class BinaryLogisticRegressionBase:
    # private:
    def __init__(self, eta, iterations=20, C=1.0):
        self.eta = eta
        self.iters = iterations
        self.C = C
        # internally we will store the weights as self.w_ to keep with sklearn conventions

    def __str__(self):
        return 'Base Binary Logistic Regression Object, Not Trainable'

    # convenience, private and static:
    @staticmethod
    def _sigmoid(theta):
        return 1/(1+np.exp(-theta))

    @staticmethod
    def _add_intercept(X):
        return np.hstack((np.ones((X.shape[0],1)),X)) # add bias term
    
    # vectorized gradient calculation with regularization using L2 Norm
    def _get_gradient(self,X,y):
        ydiff = y-self.predict_proba(X,add_bias=False).ravel() # get y difference
        gradient = np.mean(X * ydiff[:,np.newaxis], axis=0) # make ydiff a column vector and multiply through
        
        gradient = gradient.reshape(self.w_.shape)
        gradient[1:] += -2 * self.w_[1:] * self.C
        
        return gradient

    def _get_direction(self,X,y):
        return self._get_gradient(X,y)

    # public:
    def predict_proba(self, X, add_intercept=True):
        # add bias term if requested
        Xb = self._add_intercept(X) if add_intercept else X
        return self._sigmoid(Xb @ self.w_) # return the probability y=1

    def predict(self,X):
        return (self.predict_proba(X)>0.5) #return the actual prediction

class BinaryLogisticRegression(BinaryLogisticRegressionBase):
    #private:
    def __str__(self):
        if(hasattr(self,'w_')):
            return 'Binary Logistic Regression Object with coefficients:\n'+ str(self.w_) # is we have trained the object
        else:
            return 'Untrained Binary Logistic Regression Object'


    @property
    def coef_(self):
        if(hasattr(self,'w_')):
            return self.w_[1:]
        else:
            return None

    @property
    def intercept_(self):
        if(hasattr(self,'w_')):
            return self.w_[0]
        else:
            return None


    def _get_gradient(self,X,y):
        # programming \sum_i (yi-g(xi))xi
        gradient = np.zeros(self.w_.shape) # set gradient to zero
        for (xi,yi) in zip(X,y):
            # the actual update inside of sum
            gradi = (yi - self.predict_proba(xi,add_intercept=False))*xi
            # reshape to be column vector and add to gradient
            gradient += gradi.reshape(self.w_.shape)

        return gradient/float(len(y))

    # public:
    def fit(self, X, y):
        Xb = self._add_intercept(X) # add bias term
        num_samples, num_features = Xb.shape

        self.w_ = np.zeros((num_features,1)) # init weight vector to zeros

        # for as many as the max iterations
        for _ in range(self.iters):
            gradient = self._get_gradient(Xb,y)
            self.w_ += gradient*self.eta # multiply by learning rate

class VectorBinaryLogisticRegression(BinaryLogisticRegression):
    # inherit from our previous class to get same functionality
    @staticmethod
    def _sigmoid(theta):
        # increase stability, redefine sigmoid operation
        return expit(theta) #1/(1+np.exp(-theta))

    # but overwrite the gradient calculation
    def _get_gradient(self,X,y):
        ydiff = y-self.predict_proba(X,add_intercept=False).ravel() # get y difference
        gradient = np.mean(X * ydiff[:,np.newaxis], axis=0) # make ydiff a column vector and multiply through

        return gradient.reshape(self.w_.shape)

class StochasticLogisticRegression(BinaryLogisticRegression):

    # define custom line search for problem
    def __init__(self, mini_batch_size=32, **kwds):        
        self.mini_batch_size = mini_batch_size
        # but keep other keywords
        super().__init__(**kwds) # call parent initializer
    
    # stochastic "gradient" calculation 
    def _get_direction(self,X,y):
        
        # grab a subset of samples in a mini-batch
        # and calculate the gradient according to the small batch only
        idxs = np.random.choice(len(y), self.mini_batch_size)
        
        return self._get_gradient(X[idxs],y[idxs])

class HessianBinaryLogisticRegression(BinaryLogisticRegression):
    # just overwrite gradient function
    def _get_direction(self,X,y):
        g = self.predict_proba(X,add_bias=False).ravel() # get sigmoid value for all classes
        hessian = X.T @ np.diag(g*(1-g)) @ X - 2 * self.C # calculate the hessian

        gradient = self._get_gradient(X,y)
        
        return pinv(hessian) @ gradient

class LogisticRegression:

    def __init__(self, eta, iterations=20, solver='steepest', C=1.0):
        self.eta = eta
        self.iters = iterations
        # internally we will store the weights as self.w_ to keep with sklearn conventions
        self.solver = solver
        self.C = C

    def __str__(self):
        if(hasattr(self,'w_')):
            return 'MultiClass Logistic Regression Object with coefficients:\n'+ str(self.w_) # is we have trained the object
        else:
            return 'Untrained MultiClass Logistic Regression Object'

    @property
    def coef_(self):
        if(hasattr(self,'w_')):
            return self.w_[:,1:]
        else:
            return None

    @property
    def intercept_(self):
        if(hasattr(self,'w_')):
            return self.w_[:,0]
        else:
            return None

    def fit(self,X,y):
        num_samples, num_features = X.shape
        self.unique_ = np.unique(y) # get each unique class value
        num_unique_classes = len(self.unique_)
        self.classifiers_ = [] # will fill this array with binary classifiers

        for i,yval in enumerate(self.unique_): # for each unique value
            y_binary = (y==yval) # create a binary problem
            # train the binary classifier for this class

            # depending on the solver, instantiate the appropriate binary logistic regression class
            match self.solver:
                case 'steepest':
                    blr = BinaryLogisticRegression(self.eta, self.iters, C=self.C)
                case 'stochastic':
                    blr = StochasticLogisticRegression(self.eta, self.iters)
                case 'newton':
                    blr = HessianBinaryLogisticRegression(self.eta, self.iters, C=self.C)
                case _:
                    raise ValueError(f"Unknown solver: {self.solver}")

            blr.fit(X,y_binary)
            # add the trained classifier to the list
            self.classifiers_.append(blr)

        # save all the weights into one matrix, separate column for each class
        self.w_ = np.hstack([x.w_ for x in self.classifiers_]).T

    def predict_proba(self,X):
        probs = []
        for blr in self.classifiers_:
            probs.append(blr.predict_proba(X)) # get probability for each classifier

        return np.hstack(probs) # make into single matrix

    def predict(self,X):
        return self.unique_[np.argmax(self.predict_proba(X),axis=1)] # take argmax along row

lr = LogisticRegression(0.1,1500)
print(lr)

Untrained MultiClass Logistic Regression Object
